In [2]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define the path using the shortcut you just created
drive_shortcut_path = '/content/drive/MyDrive/Deep-Learning-Image-Classification'
local_extract_dir = '/content/dataset'

# 3. Copy the folder to Colab's local storage
if not os.path.exists(local_extract_dir):
    print("Copying files from Google Drive to Colab local storage. Please wait...")
    # The -r flag recursively copies all files and subfolders
    !cp -r "{drive_shortcut_path}" "{local_extract_dir}"
    print("Copy complete!")
else:
    print("Dataset already exists in local storage.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copying files from Google Drive to Colab local storage. Please wait...
Copy complete!


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Define the exact paths for your pre-split data
train_dir = '/content/dataset/Dataset/train'
val_dir = '/content/dataset/Dataset/val'
test_dir = '/content/dataset/Dataset/test'

# Standardize image dimensions for VGG16
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32

# 2. Training Generator WITH Data Augmentation and Normalization
train_datagen = ImageDataGenerator(
    rescale=1./255,             # Normalize pixels to 0-1
    rotation_range=20,          # Augmentation: Rotation
    width_shift_range=0.2,      # Augmentation: Translation
    height_shift_range=0.2,     # Augmentation: Translation
    zoom_range=0.2,             # Augmentation: Zoom
    horizontal_flip=True        # Augmentation: Horizontal Flip
)

# 3. Validation and Test Generator with ONLY Normalization (No augmentation)
test_val_datagen = ImageDataGenerator(rescale=1./255)

# 4. Load the datasets from their specific directories
print("Loading Training Data:")
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

print("\nLoading Validation Data:")
validation_generator = test_val_datagen.flow_from_directory(
    val_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False # Keep false to ensure stable validation metrics
)

print("\nLoading Test Data:")
test_generator = test_val_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False # Keep false for accurate evaluation and confusion matrices
)

print("\nClass mapping:", train_generator.class_indices)

Loading Training Data:
Found 12632 images belonging to 6 classes.

Loading Validation Data:
Found 1402 images belonging to 6 classes.

Loading Test Data:
Found 3000 images belonging to 6 classes.

Class mapping: {'buildings': 0, 'forest': 1, 'glacier': 2, 'mountain': 3, 'sea': 4, 'street': 5}


In [5]:
!ls /content/dataset/Dataset

test  train  val


In [7]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout

# Load the pretrained VGG16 model without the top classification layer
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the base model weights
base_model.trainable = False

# Build the custom architecture
model = Sequential([
    base_model,
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5), # Mitigates overfitting
    Dense(6, activation='softmax') # 6 output nodes for the 6 classes
])

model.summary()

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     6,422,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │         1,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,139,014 (80.64 MB)

 Trainable params: 6,424,326 (24.51 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [ ]:
import time
from tensorflow.keras.callbacks import EarlyStopping

# Compile the model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Stop training early if validation loss stops improving for 3 epochs
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

print("Starting VGG16 training...")
start_time = time.time()

# Train the model
history = model.fit(
    train_generator,
    epochs=15, # The early_stop callback will halt this if it converges sooner
    validation_data=validation_generator,
    callbacks=[early_stop]
)

end_time = time.time()
training_duration = end_time - start_time
print(f"\nTotal Training Time: {training_duration / 60:.2f} minutes")

# Save the trained model directly to your Shared Drive shortcut
model.save('/content/drive/MyDrive/Deep-Learning-Image-Classification/models/vgg16/vgg16_final_model.h5')
print("Model saved successfully!")

Starting VGG16 training...
Epoch 1/15
  2/395 ━━━━━━━━━━━━━━━━━━━━ 1:42:38 16s/step - accuracy: 0.1484 - loss: 3.5680